# Module 6: Hands-On Lab — Running Experiments

**Estimated time: 45 minutes**

This module walks you through actually running Model Swarms. You'll set up the environment, run a minimal experiment, and interpret the results.

## 6.1 Hardware Requirements

| Resource | Minimum | Recommended | Paper's Setup |
|----------|---------|-------------|---------------|
| GPUs | 1× 24GB (e.g., RTX 3090) | 4-5× 24GB+ | 5× A100 80GB |
| RAM | 32GB | 64GB | 128GB+ |
| Disk | 50GB free | 100GB free | 500GB+ |
| Time per run | ~2-4 hours (1 GPU) | ~30-60 min (5 GPUs) | ~20-30 min |

## 6.2 Environment Setup

Run the following commands in your terminal:

In [ ]:
# NOTE: These commands are meant to be run in a terminal.
# They are shown here for reference. Uncomment and run if needed.

# !git clone https://github.com/BunsenFeng/model_swarm.git
# !cd model_swarm && conda env create -f swarm.yml

# After activating the environment:
# !huggingface-cli login
# !cd model_swarm/initial_experts && python initial_experts.py

### Troubleshooting

- **"Can't access Gemma model"**: Accept the Gemma license on Hugging Face first
- **"CUDA out of memory"**: Add `load_in_8bit=True` to model loading in `evaluate.py`
- **"safetensors not found"**: Re-run the initial experts download script

## 6.3 Inspecting the Initial Experts

Before running the search, let's understand what we're starting with:

In [ ]:
# This code runs if you have the model_swarm repo set up.
# Otherwise, it demonstrates what the inspection would show.

import os
import json

# Simulated expert info (replace with real inspection if you have the repo)
experts = {
    "cot": {"params": "18.4M", "domain": "Chain-of-thought reasoning"},
    "code": {"params": "18.4M", "domain": "Programming"},
    "flan": {"params": "18.4M", "domain": "Instruction following"},
    "lima": {"params": "18.4M", "domain": "Curated responses"},
    "metamath": {"params": "18.4M", "domain": "Mathematics"},
    "oasst": {"params": "18.4M", "domain": "Dialogue"},
    "science": {"params": "18.4M", "domain": "Scientific text"},
    "sharegpt": {"params": "18.4M", "domain": "Conversational"},
    "ultrachat": {"params": "18.4M", "domain": "Long dialogues"},
    "wizardlm": {"params": "18.4M", "domain": "Complex instructions"},
}

print(f"{'Expert':<15} {'Parameters':<12} {'Domain'}")
print("-" * 55)
for name, info in experts.items():
    print(f"{name:<15} {info['params']:<12} {info['domain']}")

print(f"\nTotal: {len(experts)} initial experts")
print("Each is a LoRA adapter (~70MB) on top of Gemma-7B (~14GB)")

## 6.4 Running Your First Search

Here's the minimal search configuration for a resource-efficient first run:

In [ ]:
# The search command (run in terminal, shown here for reference)
search_command = """
python search.py \\
    -n my_first_search \\
    -e exact_match \\
    -d nlgraph \\
    -g 0 \\
    --inertia 0.2 \\
    --cognitive_coeff 0.3 \\
    --social_coeff 0.4 \\
    --repel_coeff 0.1 \\
    --step_length 0.8 \\
    --starting_test_set_eval 1 \\
    --fast_merge 1 \\
    --weight_randomess 1 \\
    --populate_initial_experts 1 \\
    --initial_experts_num 10 \\
    --starting_velocity_mode random \\
    --repel_term 1 \\
    --step_length_factor 0.95 \\
    --restart_stray_particles 1 \\
    --restart_patience 0.67 \\
    -p 5 \\
    -m 20 \\
    --dropK 0.3 \\
    --dropN 0.3
"""
print("Minimal search command:")
print(search_command)

print("\nKey differences from paper's settings:")
print("  - 10 particles instead of 20 (less compute)")
print("  - patience=5 instead of 10 (earlier stopping)")
print("  - max 20 iterations instead of 200")
print("  - dropK=0.3, dropN=0.3 (skip 30% of evaluations)")

## 6.5 Interpreting the Results

After the search completes, here's how to analyze the output. The code below works with simulated data — replace with your actual `utility_scratchpad.json` when available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulated search results (replace with real data from utility_scratchpad.json)
# In practice: json.load(open("search/<name>/utility_scratchpad.json"))
np.random.seed(42)

n_particles = 10
n_iterations = 15

# Simulate realistic convergence
initial_scores = np.random.uniform(0.30, 0.55, n_particles)
particle_histories = []
for i in range(n_particles):
    trajectory = [initial_scores[i]]
    for t in range(1, n_iterations + 1):
        # Particles tend to improve with noise
        improvement = np.random.uniform(-0.02, 0.05) * (1 - t/n_iterations)
        new_score = max(trajectory[-1] + improvement, trajectory[-1] - 0.03)
        trajectory.append(new_score)
    particle_histories.append(trajectory)

# Compute global best history
g_history = []
for t in range(n_iterations + 1):
    best_at_t = max(particle_histories[i][t] for i in range(n_particles))
    if g_history and best_at_t < g_history[-1]:
        g_history.append(g_history[-1])  # global best never decreases
    else:
        g_history.append(best_at_t)

print(f"Starting global best: {g_history[0]:.4f}")
print(f"Ending global best:   {g_history[-1]:.4f}")
print(f"Improvement: {g_history[-1] - g_history[0]:.4f} "
      f"({(g_history[-1] - g_history[0]) / g_history[0] * 100:.1f}%)")

### Plotting Convergence

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Global best over iterations
ax1.plot(g_history, 'b-', linewidth=2.5, label='Global best')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Utility (accuracy)')
ax1.set_title('Global Best Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: All particles' current scores over iterations
for i in range(n_particles):
    ax2.plot(particle_histories[i], alpha=0.5, label=f'Particle {i}')
ax2.plot(g_history, 'k--', linewidth=2, label='Global best', zorder=10)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Utility (accuracy)')
ax2.set_title('All Particle Trajectories')
ax2.legend(fontsize=7, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Identifying the "Diamond in the Rough"

In [ ]:
# Which particle ended up as the best?
final_bests = [max(particle_histories[i]) for i in range(n_particles)]
best_particle = np.argmax(final_bests)
initial_rank = sorted(range(n_particles), key=lambda i: -initial_scores[i])
initial_rank_of_best = initial_rank.index(best_particle) + 1

print(f"Best particle: {best_particle}")
print(f"  Initial score: {initial_scores[best_particle]:.4f}")
print(f"  Final best score: {final_bests[best_particle]:.4f}")
print(f"  Initial rank: {initial_rank_of_best} out of {n_particles}")
print(f"  Started in bottom half: {initial_rank_of_best > n_particles // 2}")

print(f"\nAll particles ranked by final best:")
for rank, i in enumerate(sorted(range(n_particles), key=lambda i: -final_bests[i])):
    marker = " ← GLOBAL BEST" if i == best_particle else ""
    print(f"  Rank {rank+1}: Particle {i} "
          f"(initial={initial_scores[i]:.3f} → final={final_bests[i]:.3f}){marker}")

## 6.6 Comparing with Baselines

In [ ]:
# Simulated baseline comparison
best_single_expert = g_history[0]
uniform_soup = np.mean(initial_scores) - 0.03  # averaging typically hurts slightly
model_swarms = g_history[-1]

baselines = {
    'Best Single Expert': best_single_expert,
    'Uniform Soup': uniform_soup,
    'Model Swarms': model_swarms,
}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax.bar(baselines.keys(), baselines.values(), color=colors, width=0.5)
ax.set_ylabel('Utility Score')
ax.set_title('Model Swarms vs. Baselines')
ax.grid(True, alpha=0.3, axis='y')

for bar, score in zip(bars, baselines.values()):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Gain over best single expert: +{model_swarms - best_single_expert:.4f}")
print(f"Gain over uniform soup: +{model_swarms - uniform_soup:.4f}")

## 6.7 Using the Resulting Model

After the search, the best model is in `search/<name>/global_best/`:

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model = "google/gemma-7b-it"
adapter_path = "search/my_first_search/global_best"

model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16)
model.load_adapter(adapter_path)
model.to("cuda:0")

tokenizer = AutoTokenizer.from_pretrained(base_model)

prompt = "What is the shortest path between node A and node D?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")
outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```

## Exercise 6.1: Run and Analyze

If you have GPU access, run the minimal search from Section 6.4. Then:
1. Plot the global best convergence curve
2. Identify which particle became the global best
3. Was it a "diamond in the rough"?
4. How many iterations ran before patience triggered?

## Exercise 6.2: Baseline Comparison

Implement and evaluate these baselines:
1. **Random soup**: Average 3 randomly selected experts
2. **Top-2 average**: Average the top 2 experts by initial score
3. **Top-3 average**: Average the top 3 experts by initial score

## Exercise 6.3: Hyperparameter Impact

Run at least 3 hyperparameter configurations. Create a comparison table.

*Your results and analysis:*



---

**Next: [Module 7 — Results, Analysis & Ablation Studies](module_07_results_analysis.ipynb)**